In [ ]:
"""
Calculate clustering metrics for a single dataset.

This script computes clustering quality metrics by:
1. Loading latent representation from a method's output
2. Performing Leiden clustering at multiple resolutions
3. Computing clustering metrics (NMI, ARI, AMI) against ground truth cell types

For reproducibility: Define your own load_dataset() function to load
adata with 'latent' representation and 'cell_type' annotations.
"""

from sklearn import metrics
import scanpy as sc
import numpy as np
import pandas as pd
import os
from typing import Dict, Tuple, Callable


def load_method_output(method, dataset):
    """
    Load the method output for a given method-dataset pair.
    
    This function should be implemented to load the results from your specific pipeline.
    
    Args:
        method (str): Name of the method
        dataset (str): Name of the dataset
        
    Returns:
        adata: AnnData object with 'latent' representation in obsm and 'cell_type', 'batch' in obs
    """
    # TODO: Implement this function to load your results
    # Example implementation:
    # return sc.read_h5ad(f'./results/{method}/{dataset}_latent.h5ad')
    raise NotImplementedError("Please implement load_method_output() for your data loading")


def _evaluate_clustering(labels_true: np.ndarray, labels_pred: np.ndarray) -> Dict[str, float]:
    results = {
        'normalized_mutual_info': metrics.normalized_mutual_info_score(labels_true, labels_pred),
        'adjusted_rand_index': metrics.adjusted_rand_score(labels_true, labels_pred),
        'adjusted_mutual_info': metrics.adjusted_mutual_info_score(labels_true, labels_pred),
    }
    return results


def bio_conservation_metrics(method, dataset, cal_neighbors=True) -> pd.DataFrame:
    """
    Compute batch metrics for a single method-dataset pair.
    
    Args:
        method (str): Name of the method
        dataset (str): Name of the dataset
        cal_neighbors (bool): Whether to calculate neighbors
        
    Returns:
        dict: Dictionary containing computed metrics
        
    Raises:
        ValueError: If required fields are missing from the AnnData object
    """
    adata = load_method_output(method, dataset)

    if adata.obsm.get('latent') is None:
        raise ValueError("Latent representation not found in adata.obsm['latent']")
    if "cell_type" not in adata.obs.columns:
        raise ValueError("Cell type labels not found in adata.obs['cell_type']")
    if "batch" not in adata.obs.columns:
        raise ValueError("Batch labels not found in adata.obs['batch']")
    

    resolutions = np.linspace(0.1, 2, 20)
    
    results = {}
    
    # Compute neighbors once
    if cal_neighbors:
        print("Calculating neighbors...")
        sc.pp.neighbors(adata, use_rep="latent")
    else:
        print("Skipping neighbor calculation. Ensure neighbors are precomputed.")
    
    # Compute clustering metrics for each resolution
    for res in resolutions:
        sc.tl.leiden(adata, resolution=res, key_added="leiden_temp")
        y_pred = adata.obs['leiden_temp'].values
        y_true = adata.obs['cell_type'].values
        result = _evaluate_clustering(y_true, y_pred)
        results[res] = result
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results).T
    results_df.index.name = 'resolution'

    return results_df


def compute_clustering_metrics(method, dataset, output_dir='./results'):
    """
    Compute and save clustering metrics for a single dataset.
    
    Args:
        method (str): Name of the method
        dataset (str): Name of the dataset
        output_dir (str): Directory to save results
        
    Returns:
        pd.DataFrame: DataFrame containing the computed metrics
    """
    try:
        results_path = f'{output_dir}/{method}/{dataset}_bio_metrics.csv'
        
        if os.path.exists(results_path):
            print(f"Results already exist at {results_path}. Skipping calculation.")
            return pd.read_csv(results_path)

        os.makedirs(f'{output_dir}/{method}/', exist_ok=True)
        
        print(f"Computing metrics for {method} - {dataset}...")
        results = bio_conservation_metrics(method, dataset)
        
        results_df = pd.DataFrame(results)
        results_df.to_csv(results_path, index=False)
        
        print(f"Results saved to {results_path}")
        print(results_df)
        
        return results_df
        
    except Exception as e:
        print(f"Error processing {method} - {dataset}: {str(e)}")
        raise




In [ ]:
METHOD = "your_method_name"  # Change this to your method name
DATASET = "your_dataset_name"  # Change this to your dataset name
OUTPUT_DIR = "./results"  # Change this to your desired output directory
# Run metrics calculation
results = compute_clustering_metrics(METHOD, DATASET, OUTPUT_DIR)
